In [40]:
# import cadquery as cq

# # 단위는 보통 mm로 가정 (CadQuery 자체는 무차원이라 일관되게 쓰면 됨)
# A = 30.0  # overall height
# B = 25.0  # overall width
# C = 10.0  # stack depth (thickness)
# E = 18.0  # window height
# F = 12.0  # window width

# # 외형(실체)
# outer = cq.Workplane("XY").box(B, A, C, centered=(True, True, True))

# # 윈도우(빼낼 부피): 정중앙 관통 구멍
# window = cq.Workplane("XY").box(F, E, C + 1.0, centered=(True, True, True))

# core = outer.cut(window)

In [41]:
from __future__ import annotations

import os
from pathlib import Path

from pyaedt import Desktop, Maxwell3d


class MaxwellEddyCurrentSession:
    def __init__(self, project_path: Path, design_name: str) -> None:
        self.project_path = project_path
        self.design_name = design_name
        self.desktop = self._start_desktop()
        self.m3d = Maxwell3d(
            project=str(self.project_path),
            design=self.design_name,
            solution_type="EddyCurrent",
            non_graphical=True,
            new_desktop=False,
        )

    def _start_desktop(self) -> Desktop:
        version = os.getenv("AEDT_VERSION")
        if version:
            return Desktop(version=version, non_graphical=True, new_desktop=False)
        return Desktop(non_graphical=False, new_desktop=False)

    def save(self) -> None:
        self.m3d.save_project()

    def close(self) -> None:
        self.desktop.close_desktop()



In [42]:
project_path = Path("/home/harry/Projects/AedtProjects/byPeetsFea").resolve() / "maxwell_eddy_current.aedt"
design_name = "EddyCurrentDesign"

session = MaxwellEddyCurrentSession(project_path, design_name)
session.save()
# session.desktop.release_desktop(close_on_exit=False, close_projects=False)


PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO: PyAEDT version 0.24.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file /tmp/pyaedt_harry_9af3532b-4d21-45ed-ba6b-23d6d9e9cd90.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Found active AEDT gRPC session on port 38267.
PyAEDT INFO: Connecting to AEDT gRPC session on port 38267.
PyAEDT INFO: AEDT installation Path /opt/ansys_inc/v252/AnsysEM
PyAEDT INFO: Client application successfully started.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO:

In [43]:
# PyAEDT parametric UF-core (U-shape) using D (inner leg thickness)

from sympy import symbols

A_val = 120.0  # overall height (Y)
B_val = 117.0  # overall width  (X)
C_val = 40.0   # stack depth / thickness (Z)

D_val = 30.0   # inner leg thickness (remaining wall on the closed side, +X side)
E_val = 59.0   # window height (Y)

z_oversize_val = 1.0

A, B, C, D, E, z_oversize = symbols("A B C D E z_oversize")

# Derived: window width from open side to inner-leg face
F = B - D  # distance from open outer face (x=-B/2) to inner leg

modeler = session.m3d.modeler
from ansys.aedt.core.modeler.modeler_3d import Modeler3D
assert isinstance(modeler, Modeler3D)
modeler.model_units = "mm"

# Push numeric values into AEDT so expressions resolve
for name, val in {
    "A": A_val,
    "B": B_val,
    "C": C_val,
    "D": D_val,
    "E": E_val,
    "z_oversize": z_oversize_val,
}.items():
    session.m3d[name] = f"{val}mm"

# (Optional) sanity check
F_val = B_val - D_val
if F_val <= 0:
    raise ValueError(
        f"Invalid dimensions: B({B_val}) must be > D({D_val}). Computed F={F_val}."
    )


def s(expr) -> str:
    return str(expr)


# Outer block (centered at origin)
outer = modeler.create_box(
    [s(-B / 2), s(-A / 2), s(-C / 2)],
    [s(B), s(A), s(C)],
    name="outer",
)

# Window block: touches the open outer face (x = -B/2) so the opening is to outside (UF/U-shape)
# Oversize in Z to guarantee a clean through-cut
window = modeler.create_box(
    [s(-B / 2), s(-E / 2), s(-(C + z_oversize) / 2)],  # start at open outer face
    [s(F), s(E), s(C + z_oversize)],                   # go inward by F, leaving inner leg thickness D
    name="window",
)

# Subtract to form UF core
core = modeler.subtract(outer, [window], keep_originals=False)

# (Optional) rename final object
# core.name = "UF_core"


PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec


In [44]:
session.desktop.release_desktop(close_on_exit=False, close_projects=False)


PyAEDT INFO: Desktop has been released.


True